In [ ]:
!pip install -q --no-index --find-links=/kaggle/input/mabe-package xgboost==3.1.1

In [ ]:
#训练的模型
!mkdir -p /kaggle/tmp/results
!cp -r /kaggle/input/mabe-exp-1209/exp_1209_new_feature/content/exp_1209_new_feature/results/* /kaggle/tmp/results/
!cp -rn /kaggle/input/mabe-exp-1210-v6/results/* /kaggle/tmp/results/


In [ ]:
import gc
import itertools
import re
import sys
from collections import defaultdict
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import polars as pl
import xgboost as xgb
from tqdm.auto import tqdm

In [ ]:
# const
INPUT_DIR = Path("/kaggle/input/MABe-mouse-behavior-detection")
TRAIN_TRACKING_DIR = INPUT_DIR / "train_tracking"
TRAIN_ANNOTATION_DIR = INPUT_DIR / "train_annotation"
TEST_TRACKING_DIR = INPUT_DIR / "test_tracking"

WORKING_DIR = Path("/kaggle/tmp")

INDEX_COLS = [
    "video_id",
    "agent_mouse_id",
    "target_mouse_id",
    "video_frame",
]

BODY_PARTS = [
    "ear_left",
    "ear_right",
    "nose",
    "neck",
    "body_center",
    "lateral_left",
    "lateral_right",
    "hip_left",
    "hip_right",
    "tail_base",
    "tail_tip",
]

SELF_BEHAVIORS = [
    "biteobject",
    "climb",
    "dig",
    "exploreobject",
    "freeze",
    "genitalgroom",
    "huddle",
    "rear",
    "rest",
    "run",
    "selfgroom",
]

PAIR_BEHAVIORS = [
    "allogroom",
    "approach",
    "attack",
    "attemptmount",
    "avoid",
    "chase",
    "chaseattack",
    "defend",
    "disengage",
    "dominance",
    "dominancegroom",
    "dominancemount",
    "ejaculate",
    "escape",
    "flinch",
    "follow",
    "intromit",
    "mount",
    "reciprocalsniff",
    "shepherd",
    "sniff",
    "sniffbody",
    "sniffface",
    "sniffgenital",
    "submit",
    "tussle",
]

In [ ]:
# read data
test_dataframe = pl.read_csv(INPUT_DIR / "test.csv")

In [ ]:
# preprocess behavior labels
test_behavior_dataframe = (
    test_dataframe.filter(pl.col("behaviors_labeled").is_not_null())
    .select(
        pl.col("lab_id"),
        pl.col("video_id"),
        pl.col("behaviors_labeled").map_elements(eval, return_dtype=pl.List(pl.Utf8)).alias("behaviors_labeled_list"),
    )
    .explode("behaviors_labeled_list")
    .rename({"behaviors_labeled_list": "behaviors_labeled_element"})
    .select(
        pl.col("lab_id"),
        pl.col("video_id"),
        pl.col("behaviors_labeled_element").str.split(",").list[0].str.replace_all("'", "").alias("agent"),
        pl.col("behaviors_labeled_element").str.split(",").list[1].str.replace_all("'", "").alias("target"),
        pl.col("behaviors_labeled_element").str.split(",").list[2].str.replace_all("'", "").alias("behavior"),
    )
)

test_self_behavior_dataframe = test_behavior_dataframe.filter(pl.col("behavior").is_in(SELF_BEHAVIORS))
test_pair_behavior_dataframe = test_behavior_dataframe.filter(pl.col("behavior").is_in(PAIR_BEHAVIORS))

In [ ]:

# 定义用于计算二阶形态特征的关键部位对
# 这些对的变化能反映具体的单体行为（如伸展、蜷缩、理毛）
MORPH_PAIRS = [
    ("nose", "tail_base"),     # 身体全长 (Rear, Huddle)
    ("nose", "body_center"),   # 前半身伸缩 (Investigate)
    ("nose", "hip_left"),      # 扭头 (Grooming)
    ("nose", "hip_right"),     # 扭头 (Grooming)
    ("lateral_left", "lateral_right") # 身体宽度 (呼吸/姿态挤压)
]

def make_self_features(
    metadata: dict,
    tracking: pl.DataFrame,
) -> pl.DataFrame:
    fps = metadata["frames_per_second"]
    pix_per_cm = metadata["pix_per_cm_approx"]
    video_id = metadata["video_id"]

    # 场地参数
    arena_w = metadata.get("arena_width_cm", 50.0) * pix_per_cm
    arena_h = metadata.get("arena_height_cm", 50.0) * pix_per_cm

    start_frame = tracking.select(pl.col("video_frame").min()).item()
    end_frame = tracking.select(pl.col("video_frame").max()).item()

    # --- 辅助计算函数 ---
    def get_w(period_ms):
        return max(1, int(round(period_ms * fps / 1000.0)))

    def calc_dist(x1, y1, x2, y2):
        if isinstance(x1, str): x1 = pl.col(x1)
        if isinstance(y1, str): y1 = pl.col(y1)
        if isinstance(x2, str): x2 = pl.col(x2)
        if isinstance(y2, str): y2 = pl.col(y2)
        return ((x1 - x2).pow(2) + (y1 - y2).pow(2)).sqrt() / pix_per_cm

    def calc_speed(bp, period_ms):
        w = get_w(period_ms)
        d = ((pl.col(f"agent_x_{bp}").diff()).pow(2) + (pl.col(f"agent_y_{bp}").diff()).pow(2)).sqrt()
        return (d / pix_per_cm * fps).rolling_mean(window_size=w, center=True)

        # 频域/高频能量特征 ---
    def calc_high_freq_energy(bp, period_ms):
        """计算高频震动能量 (代替 FFT): 加速度的绝对值在窗口内的均值"""
        w = get_w(period_ms)
        # 1. 速度
        vx = pl.col(f"agent_x_{bp}").diff()
        vy = pl.col(f"agent_y_{bp}").diff()
        # 2. 加速度 (变化率)
        acc_mag = (vx.diff().pow(2) + vy.diff().pow(2)).sqrt()
        # 3. 能量 (窗口均值)
        return (acc_mag / pix_per_cm * (fps**2)).rolling_mean(window_size=w, center=True)

    def calc_accel(speed_col_name, period_ms):
        w = get_w(period_ms)
        return (pl.col(speed_col_name).diff().abs() * fps).rolling_mean(window_size=w, center=True)

    def calc_heading():
        return pl.arctan2(
            pl.col("agent_y_nose") - pl.col("agent_y_tail_base"),
            pl.col("agent_x_nose") - pl.col("agent_x_tail_base")
        )

    def calc_angular_vel(heading_col_name, period_ms):
        w = get_w(period_ms)
        diff = pl.col(heading_col_name).diff()
        # 处理角度跳变 (-pi 到 pi)
        diff_wrapped = (diff + np.pi).mod(2 * np.pi) - np.pi
        return (diff_wrapped.abs() * fps).rolling_mean(window_size=w, center=True)

    # --- 挖掘特征函数 ---
    def calc_wall_dist():
        dist_x = pl.min_horizontal([pl.col("agent_x_body_center"), pl.lit(arena_w) - pl.col("agent_x_body_center")])
        dist_y = pl.min_horizontal([pl.col("agent_y_body_center"), pl.lit(arena_h) - pl.col("agent_y_body_center")])
        return pl.min_horizontal([dist_x, dist_y]) / pix_per_cm

    def calc_spine_angle():
        v1x = pl.col("agent_x_nose") - pl.col("agent_x_body_center")
        v1y = pl.col("agent_y_nose") - pl.col("agent_y_body_center")
        v2x = pl.col("agent_x_tail_base") - pl.col("agent_x_body_center")
        v2y = pl.col("agent_y_tail_base") - pl.col("agent_y_body_center")
        dot = v1x * v2x + v1y * v2y
        mag = (v1x.pow(2) + v1y.pow(2)).sqrt() * (v2x.pow(2) + v2y.pow(2)).sqrt()
        return (dot / (mag + 1e-6))

    def calc_tortuosity(period_ms):
        w = get_w(period_ms)
        step_dist = ((pl.col("agent_x_body_center").diff()).pow(2) + (pl.col("agent_y_body_center").diff()).pow(2)).sqrt()
        path_len = step_dist.rolling_sum(window_size=w, center=True)
        disp = ((pl.col("agent_x_body_center").diff(n=w)).pow(2) + (pl.col("agent_y_body_center").diff(n=w)).pow(2)).sqrt()
        return path_len / (disp + 1e-6)

    def calc_jitter(period_ms):
        w = get_w(period_ms)
        # 计算 鼻尖相对于身体中心 的高频颤动
        rel_x = pl.col("agent_x_nose") - pl.col("agent_x_body_center")
        rel_y = pl.col("agent_y_nose") - pl.col("agent_y_body_center")
        rel_speed = ((rel_x.diff()).pow(2) + (rel_y.diff()).pow(2)).sqrt()
        return (rel_speed / pix_per_cm * fps).rolling_mean(window_size=w, center=True)

    def calc_polygon_area():
        # 使用鞋带公式 (Shoelace formula) 计算由 nose, ears, hips, tail 构成的多边形面积
        # 简化版：Nose -> EarL -> HipL -> TailBase -> HipR -> EarR -> Nose
        x_cols = ["agent_x_nose", "agent_x_ear_left", "agent_x_hip_left", "agent_x_tail_base", "agent_x_hip_right", "agent_x_ear_right"]
        y_cols = ["agent_y_nose", "agent_y_ear_left", "agent_y_hip_left", "agent_y_tail_base", "agent_y_hip_right", "agent_y_ear_right"]

        # 构建表达式 (x1y2 - y1x2) + ...
        expr = pl.lit(0)
        for i in range(len(x_cols)):
            j = (i + 1) % len(x_cols)
            expr = expr + (pl.col(x_cols[i]) * pl.col(y_cols[j]) - pl.col(y_cols[i]) * pl.col(x_cols[j]))

        return (expr.abs() * 0.5) / (pix_per_cm * pix_per_cm)

    # --- 数据准备 ---
    group_centroid = tracking.group_by("video_frame").agg(
        pl.col("x").mean().alias("group_center_x"),
        pl.col("y").mean().alias("group_center_y")
    ).sort("video_frame")

    pivot = tracking.pivot(
        on=["bodypart"],
        index=["video_frame", "mouse_id"],
        values=["x", "y"],
    ).sort(["mouse_id", "video_frame"])

    n_mice = (
        (metadata["mouse1_strain"] is not None)
        + (metadata["mouse2_strain"] is not None)
        + (metadata["mouse3_strain"] is not None)
        + (metadata["mouse4_strain"] is not None)
    )
    mice_ids = range(1, n_mice + 1)
    pivot_trackings = {mouse_id: pivot.filter(pl.col("mouse_id") == mouse_id) for mouse_id in mice_ids}

    result = []

    for agent_mouse_id in mice_ids:
        # A. Skeleton
        result_element = pl.DataFrame(
            {
                "video_id": video_id,
                "agent_mouse_id": agent_mouse_id,
                "target_mouse_id": -1,
                "video_frame": pl.arange(start_frame, end_frame + 1, eager=True),
            },
            schema={
                "video_id": pl.Int32,
                "agent_mouse_id": pl.Int8,
                "target_mouse_id": pl.Int8,
                "video_frame": pl.Int32,
            },
        )

        # B. Prepare Data
        curr_pivot = pivot_trackings[agent_mouse_id]
        pivot_cols = curr_pivot.columns
        exprs = [pl.col("video_frame")]
        missing_cols = []

        for bp in BODY_PARTS:
            if f"x_{bp}" in pivot_cols:
                exprs.append(pl.col(f"x_{bp}").alias(f"agent_x_{bp}"))
            else:
                missing_cols.append(pl.lit(None).cast(pl.Float32).alias(f"agent_x_{bp}"))
            if f"y_{bp}" in pivot_cols:
                exprs.append(pl.col(f"y_{bp}").alias(f"agent_y_{bp}"))
            else:
                missing_cols.append(pl.lit(None).cast(pl.Float32).alias(f"agent_y_{bp}"))

        agent_df = curr_pivot.select(exprs)
        if missing_cols:
            agent_df = agent_df.with_columns(missing_cols)

        ##
        agent_df = preprocess_tracking(agent_df, metadata)

        agent_df = agent_df.join(group_centroid, on="video_frame", how="left")

        # 预计算向量
        agent_df = agent_df.with_columns(
            calc_heading().alias("temp_heading"),
            calc_dist("agent_x_nose", "agent_y_nose", "agent_x_tail_base", "agent_y_tail_base").alias("temp_len")
        )

        # C. Select Features (基础特征 + 依赖坐标的二阶特征)
        # 我们将 "Deformation Velocity" 和 "Posture Jitter" 移到这里，因为它们需要 agent_x/y 坐标
        features = agent_df.select(
            pl.col("video_frame"),
            pl.lit(agent_mouse_id).alias("agent_mouse_id"),
            pl.lit(-1).alias("target_mouse_id"),

            # 1. 身体各部位两两之间的距离
            *[
                calc_dist(f"agent_x_{bp1}", f"agent_y_{bp1}", f"agent_x_{bp2}", f"agent_y_{bp2}")
                .alias(f"aa__{bp1}__{bp2}__distance")
                for bp1, bp2 in itertools.combinations(BODY_PARTS, 2)
            ],

            # 2. 各部位速度
            *[
                calc_speed(bp, ms).alias(f"agent__{bp}__speed_{ms}ms")
                for bp in ["ear_left", "ear_right", "tail_base", "body_center", "nose"]
                for ms in [200, 500, 1000]
            ],

            # 3. 高级挖掘特征
            pl.col("temp_heading").alias("agent__heading"),
            pl.col("temp_len").alias("agent__body_length"),
            (pl.col("temp_len") / (calc_dist("agent_x_ear_left", "agent_y_ear_left", "agent_x_ear_right", "agent_y_ear_right") + 1e-6)).alias("agent__elongation"),

            calc_wall_dist().alias("agent__dist_to_wall"), # 趋墙性
            calc_spine_angle().alias("agent__spine_angle"), # 姿态
            calc_dist("agent_x_body_center", "agent_y_body_center", "group_center_x", "group_center_y").alias("agent__dist_to_group_center"), # 社交上下文
            calc_tortuosity(500).alias("agent__tortuosity_500ms"), # 路径曲折度
            calc_jitter(200).alias("agent__nose_jitter_200ms"), # 微动作
            calc_angular_vel("temp_heading", 200).alias("agent__ang_vel_200ms"), # 角速度

            # --- C. 形变速度 (Deformation Velocity) ---
            # 依赖原始坐标，必须在此计算
            *[
                (calc_dist(f"agent_x_{bp1}", f"agent_y_{bp1}", f"agent_x_{bp2}", f"agent_y_{bp2}")
                 .diff() * fps / pix_per_cm)
                .rolling_mean(get_w(200), center=True)
                .alias(f"agent__{bp1}__{bp2}__stretch_vel_200ms")
                for bp1, bp2 in MORPH_PAIRS
            ],

            # --- D. 姿态抖动/剧烈度 (Posture Jitter) ---
            # 依赖原始坐标，必须在此计算
            *[
                calc_dist(f"agent_x_{bp1}", f"agent_y_{bp1}", f"agent_x_{bp2}", f"agent_y_{bp2}")
                .rolling_std(get_w(500), center=True)
                .alias(f"agent__{bp1}__{bp2}__pose_jitter_500ms")
                for bp1, bp2 in MORPH_PAIRS
            ],

            calc_polygon_area().alias("agent__body_area"),

            ## 频域/能量特征 (Grooming, Scratching 关键)
            calc_high_freq_energy("nose", 200).alias("agent__nose__energy_200ms"),
            calc_high_freq_energy("ear_left", 200).alias("agent__ear_l__energy_200ms"),
            calc_high_freq_energy("tail_base", 200).alias("agent__tail__energy_200ms"),
        )

        # 4. 二次加工特征 (依赖步骤3生成的新特征)
        features = features.with_columns(
            # --- A. 加速度 (Acceleration) ---
            calc_accel("agent__body_center__speed_500ms", 500).alias("agent__body_center__accel_500ms"),
            calc_accel("agent__nose__speed_200ms", 200).alias("agent__nose__accel_200ms"),

            # --- B. 速度的波动率 (Speed Stability) ---
            pl.col("agent__body_center__speed_200ms").rolling_std(get_w(500), center=True).alias("agent__speed_std_500ms"),

            # --- E. 复合特征交互 ---
            # 1. Grooming Ratio: (局部抖动 / (整体速度 + 1))
            (pl.col("agent__nose_jitter_200ms").fill_null(0) / (pl.col("agent__body_center__speed_200ms").fill_null(0) + 1.0))
            .alias("agent__interaction__grooming_ratio"),

            # 2. Agility Index: 速度 * 角速度
            (pl.col("agent__body_center__speed_200ms").fill_null(0) * pl.col("agent__ang_vel_200ms").fill_null(0))
            .alias("agent__interaction__agility_index"),

            # 3. Rearing Score Estimate: 身体长 / (速度 + 1)
            (pl.col("agent__body_length").fill_null(0) / (pl.col("agent__body_center__speed_500ms").fill_null(0) + 1.0))
            .alias("agent__interaction__rear_score"),
        )

        # [NEW] 2. 简单的时间平滑 (Long Context)
        # 计算关键特征在 1秒 (30帧) 窗口内的均值和标准差
        # 这样模型知道"它是刚刚开始跑，还是已经跑了一会儿了"
        long_window = get_w(1000)
        cols_to_smooth = [c for c in features.columns if "speed" in c or "energy" in c or "area" in c]

        features = features.with_columns([
            pl.col(c).rolling_mean(long_window, center=True).alias(f"{c}_avg_1s") for c in cols_to_smooth
        ] + [
            pl.col(c).rolling_std(long_window, center=True).alias(f"{c}_std_1s") for c in cols_to_smooth
        ])
        features = features.with_columns([
            pl.col("agent__body_center__speed_500ms").shift(15).alias("agent__speed_lag_15"), # 约0.5秒前
            pl.col("agent__body_center__speed_500ms").shift(-15).alias("agent__speed_lead_15"), # 约0.5秒后

            pl.col("agent__body_center__speed_500ms").shift(10).alias("agent__speed_lag_10"), # 约0.5秒前
            pl.col("agent__body_center__speed_500ms").shift(-10).alias("agent__speed_lead_10"), # 约0.5秒后

            pl.col("agent__body_center__speed_500ms").shift(5).alias("agent__speed_lag_5"), # 约0.5秒前
            pl.col("agent__body_center__speed_500ms").shift(-5).alias("agent__speed_lead_5"), # 约0.5秒后

            # 你也可以加加速度的滞后
            pl.col("agent__body_center__accel_500ms").shift(15).alias("agent__accel_lag_15"),
            pl.col("agent__body_center__accel_500ms").shift(-15).alias("agent__accel_lead_15"),

            pl.col("agent__body_center__accel_500ms").shift(10).alias("agent__accel_lag_10"),
            pl.col("agent__body_center__accel_500ms").shift(-10).alias("agent__accel_lead_10"),

            pl.col("agent__body_center__accel_500ms").shift(5).alias("agent__accel_lag_5"),
            pl.col("agent__body_center__accel_500ms").shift(-5).alias("agent__accel_lead_5"),
        ])
        result_element = result_element.join(
            features,
            on=["video_frame", "agent_mouse_id", "target_mouse_id"],
            how="left",
        )
        result.append(result_element)

    return pl.concat(result, how="vertical")

def make_pair_features(
    metadata: dict,
    tracking: pl.DataFrame,
) -> pl.DataFrame:

    fps = metadata["frames_per_second"]
    pix_per_cm = metadata["pix_per_cm_approx"]
    video_id = metadata["video_id"]

    # --- 辅助函数 ---
    def get_w(period_ms):
        return max(1, int(round(period_ms * fps / 1000.0)))

    def calc_dist(x1, y1, x2, y2):
        if isinstance(x1, str): x1 = pl.col(x1)
        if isinstance(y1, str): y1 = pl.col(y1)
        if isinstance(x2, str): x2 = pl.col(x2)
        if isinstance(y2, str): y2 = pl.col(y2)
        return ((x1 - x2).pow(2) + (y1 - y2).pow(2)).sqrt() / pix_per_cm

    def body_parts_distance(agent_or_target_1, body_part_1, agent_or_target_2, body_part_2):
        return calc_dist(
            f"{agent_or_target_1}_x_{body_part_1}", f"{agent_or_target_1}_y_{body_part_1}",
            f"{agent_or_target_2}_x_{body_part_2}", f"{agent_or_target_2}_y_{body_part_2}"
        )

    def body_part_speed(agent_or_target, body_part, period_ms):
        w = get_w(period_ms)
        return (
            ((pl.col(f"{agent_or_target}_x_{body_part}").diff()).pow(2)
             + (pl.col(f"{agent_or_target}_y_{body_part}").diff()).pow(2)).sqrt()
            / pix_per_cm * fps
        ).rolling_mean(window_size=w, center=True)

    def calc_heading(prefix):
        return pl.arctan2(
            pl.col(f"{prefix}_y_nose") - pl.col(f"{prefix}_y_tail_base"),
            pl.col(f"{prefix}_x_nose") - pl.col(f"{prefix}_x_tail_base")
        )

    def calc_relative_heading():
        agent_heading = calc_heading("agent")
        heading_to_target = pl.arctan2(
            pl.col("target_y_body_center") - pl.col("agent_y_body_center"),
            pl.col("target_x_body_center") - pl.col("agent_x_body_center")
        )
        diff = agent_heading - heading_to_target
        return (diff + np.pi).mod(2 * np.pi) - np.pi

    def calc_mutual_facing():
        agent_heading = calc_heading("agent")
        target_heading = calc_heading("target")
        diff = (agent_heading - target_heading).abs()
        return (diff - np.pi).abs()

    def calc_approach_speed(period_ms):
        w = get_w(period_ms)
        dist = body_parts_distance("agent", "body_center", "target", "body_center")
        return (dist.diff() * fps).rolling_mean(w, center=True)

    def calc_relative_speed():
        agent_vx = pl.col("agent_x_body_center").diff()
        agent_vy = pl.col("agent_y_body_center").diff()
        target_vx = pl.col("target_x_body_center").diff()
        target_vy = pl.col("target_y_body_center").diff()
        rel_vx = agent_vx - target_vx
        rel_vy = agent_vy - target_vy
        return (rel_vx.pow(2) + rel_vy.pow(2)).sqrt() / pix_per_cm * fps

    def elongation(prefix):
        d1 = body_parts_distance(prefix, "nose", prefix, "tail_base")
        d2 = body_parts_distance(prefix, "ear_left", prefix, "ear_right")
        return d1 / (d2 + 1e-6)

    def body_angle(prefix):
        v1x = pl.col(f"{prefix}_x_nose") - pl.col(f"{prefix}_x_body_center")
        v1y = pl.col(f"{prefix}_y_nose") - pl.col(f"{prefix}_y_body_center")
        v2x = pl.col(f"{prefix}_x_tail_base") - pl.col(f"{prefix}_x_body_center")
        v2y = pl.col(f"{prefix}_y_tail_base") - pl.col(f"{prefix}_y_body_center")
        return (v1x * v2x + v1y * v2y) / ((v1x.pow(2) + v1y.pow(2)).sqrt() * (v2x.pow(2) + v2y.pow(2)).sqrt() + 1e-6)

    def calc_egocentric(agent, target):
        ax = pl.col(f"{agent}_x_nose") - pl.col(f"{agent}_x_tail_base")
        ay = pl.col(f"{agent}_y_nose") - pl.col(f"{agent}_y_tail_base")
        a_len = (ax.pow(2) + ay.pow(2)).sqrt() + 1e-6
        tx = pl.col(f"{target}_x_body_center") - pl.col(f"{agent}_x_body_center")
        ty = pl.col(f"{target}_y_body_center") - pl.col(f"{agent}_y_body_center")
        longitudinal = (ax * tx + ay * ty) / a_len
        lateral = (ax * ty - ay * tx) / a_len
        return [
            (longitudinal / pix_per_cm).alias(f"{agent}_to_{target}__long_dist"),
            (lateral / pix_per_cm).alias(f"{agent}_to_{target}__lat_dist_signed"),
            (lateral / pix_per_cm).abs().alias(f"{agent}_to_{target}__lat_dist"),
        ]

    def calc_facing_angle(agent, target):
        ah_x = pl.col(f"{agent}_x_nose") - pl.col(f"{agent}_x_tail_base")
        ah_y = pl.col(f"{agent}_y_nose") - pl.col(f"{agent}_y_tail_base")
        at_x = pl.col(f"{target}_x_body_center") - pl.col(f"{agent}_x_body_center")
        at_y = pl.col(f"{target}_y_body_center") - pl.col(f"{agent}_y_body_center")
        dot = ah_x * at_x + ah_y * at_y
        mag = (ah_x.pow(2) + ah_y.pow(2)).sqrt() * (at_x.pow(2) + at_y.pow(2)).sqrt()
        return dot / (mag + 1e-6)

    # --- 数据准备 ---
    n_mice = sum([metadata[f"mouse{i}_strain"] is not None for i in range(1, 5)])
    start_frame = tracking.select(pl.col("video_frame").min()).item()
    end_frame = tracking.select(pl.col("video_frame").max()).item()

    pivot = tracking.pivot(
        on=["bodypart"],
        index=["video_frame", "mouse_id"],
        values=["x", "y"],
    ).sort(["mouse_id", "video_frame"])

    pivot_trackings = {
        mouse_id: pivot.filter(pl.col("mouse_id") == mouse_id)
        for mouse_id in range(1, n_mice + 1)
    }

    result = []

    for agent_mouse_id, target_mouse_id in itertools.permutations(range(1, n_mice + 1), 2):
        result_element = pl.DataFrame({
            "video_id": video_id,
            "agent_mouse_id": agent_mouse_id,
            "target_mouse_id": target_mouse_id,
            "video_frame": pl.arange(start_frame, end_frame + 1, eager=True),
        }, schema={
            "video_id": pl.Int32,
            "agent_mouse_id": pl.Int8,
            "target_mouse_id": pl.Int8,
            "video_frame": pl.Int32,
        })

        merged_pivot = (
            pivot_trackings[agent_mouse_id]
            .select(pl.col("video_frame"), pl.exclude("video_frame").name.prefix("agent_"))
            .join(
                pivot_trackings[target_mouse_id].select(
                    pl.col("video_frame"), pl.exclude("video_frame").name.prefix("target_")
                ),
                on="video_frame",
                how="inner",
            )
        )

        columns = merged_pivot.columns
        merged_pivot = merged_pivot.with_columns(
            *[pl.lit(None).cast(pl.Float32).alias(f"agent_x_{bp}")
              for bp in BODY_PARTS if f"agent_x_{bp}" not in columns],
            *[pl.lit(None).cast(pl.Float32).alias(f"agent_y_{bp}")
              for bp in BODY_PARTS if f"agent_y_{bp}" not in columns],
            *[pl.lit(None).cast(pl.Float32).alias(f"target_x_{bp}")
              for bp in BODY_PARTS if f"target_x_{bp}" not in columns],
            *[pl.lit(None).cast(pl.Float32).alias(f"target_y_{bp}")
              for bp in BODY_PARTS if f"target_y_{bp}" not in columns],
        )

        merged_pivot = preprocess_tracking(merged_pivot, metadata)

        # ========== 第一阶段：基础特征 ==========
        features = merged_pivot.with_columns(
            pl.lit(agent_mouse_id).alias("agent_mouse_id"),
            pl.lit(target_mouse_id).alias("target_mouse_id"),
        ).select(
            pl.col("video_frame"),
            pl.col("agent_mouse_id"),
            pl.col("target_mouse_id"),

            # --- 2. 全部部位距离（用于兼容性）---
            *[
                body_parts_distance("agent", agent_bp, "target", target_bp)
                .alias(f"at__{agent_bp}__{target_bp}__dist")
                for agent_bp, target_bp in itertools.product(BODY_PARTS, repeat=2)
            ],

            # --- 3. 最小距离特征 ---
            pl.min_horizontal([
                body_parts_distance("agent", "nose", "target", bp)
                for bp in ["nose", "ear_left", "ear_right", "neck", "body_center"]
            ]).alias("agent_nose__min_dist_to_target_head"),

            pl.min_horizontal([
                body_parts_distance("agent", "nose", "target", bp)
                for bp in ["hip_left", "hip_right", "tail_base"]
            ]).alias("agent_nose__min_dist_to_target_rear"),

            # --- 4. 速度特征 ---
            *[
                body_part_speed("agent", bp, ms).alias(f"agent__{bp}__speed_{ms}ms")
                for bp in ["body_center", "nose", "tail_base", "ear_left", "ear_right"]
                for ms in [200, 500, 1000]
            ],
            *[
                body_part_speed("target", bp, ms).alias(f"target__{bp}__speed_{ms}ms")
                for bp in ["body_center", "nose", "tail_base", "ear_left", "ear_right"]
                for ms in [200, 500, 1000]
            ],

            # --- 5. 相对速度 ---
            calc_relative_speed().alias("relative_speed"),

            # --- 6. 朝向特征 ---
            calc_heading("agent").alias("agent_heading"),
            calc_heading("target").alias("target_heading"),
            calc_relative_heading().alias("agent_relative_heading"),
            calc_mutual_facing().alias("mutual_facing_angle"),

            # --- 7. 自身中心坐标系 ---
            *calc_egocentric("agent", "target"),
            *calc_egocentric("target", "agent"),

            # --- 8. Facing Score ---
            calc_facing_angle("agent", "target").alias("agent_facing_score"),
            calc_facing_angle("target", "agent").alias("target_facing_score"),

            # --- 9. 接近速度 ---
            calc_approach_speed(200).alias("approach_speed_200ms"),
            calc_approach_speed(500).alias("approach_speed_500ms"),

            # --- 10. 姿态特征 ---
            elongation("agent").alias("agent_elongation"),
            elongation("target").alias("target_elongation"),
            body_angle("agent").alias("agent_body_angle"),
            body_angle("target").alias("target_body_angle"),
        )

        # ========== 第二阶段：派生特征（依赖第一阶段的列）==========
        features = features.with_columns(
            # --- 速度差 ---
            (pl.col("agent__body_center__speed_200ms") - pl.col("target__body_center__speed_200ms"))
            .alias("speed_diff_200ms"),
            (pl.col("agent__body_center__speed_500ms") - pl.col("target__body_center__speed_500ms"))
            .alias("speed_diff_500ms"),

            # --- 速度比 ---
            (pl.col("agent__body_center__speed_200ms") / (pl.col("target__body_center__speed_200ms") + 0.1))
            .alias("speed_ratio_200ms"),

            # --- Contact score ---
            (pl.col("at__body_center__body_center__dist") < 5).cast(pl.Float32)
            .alias("in_contact"),

            # --- Face-to-face score ---
            ((pl.col("mutual_facing_angle") < 0.5) &
             (pl.col("at__nose__nose__dist") < 5)).cast(pl.Float32)
            .alias("face_to_face"),
        )

        # ========== 第三阶段：复合交互特征（依赖第二阶段的列）==========
        features = features.with_columns(
            # Chase score
            ((pl.col("speed_diff_200ms") > 0).cast(pl.Float32) * 0.33 +
             (pl.col("agent_facing_score") > 0.5).cast(pl.Float32) * 0.33 +
             (pl.col("approach_speed_500ms") < 0).cast(pl.Float32) * 0.33)
            .alias("chase_score"),

            # Flee score
            ((pl.col("speed_diff_200ms") < 0).cast(pl.Float32) * 0.33 +
             (pl.col("target_facing_score") < 0).cast(pl.Float32) * 0.33 +
             (pl.col("approach_speed_500ms") > 0).cast(pl.Float32) * 0.33)
            .alias("flee_score"),

            # Sniffing score
            ((pl.col("agent_nose__min_dist_to_target_head") < 3).cast(pl.Float32) +
             (pl.col("agent_nose__min_dist_to_target_rear") < 3).cast(pl.Float32)) *
            (1 / (pl.col("agent__body_center__speed_200ms") + 1))
            .alias("sniff_score"),
        )

        # ========== 第四阶段：时间窗口统计特征 ==========
        long_window = get_w(1000)

        cols_to_smooth = [
            "at__body_center__body_center__dist",
            "approach_speed_500ms",
            "speed_diff_200ms",
            "chase_score",
        ]

        features = features.with_columns([
            pl.col(c).rolling_mean(long_window, center=True).alias(f"{c}_avg_1s")
            for c in cols_to_smooth
        ] + [
            pl.col(c).rolling_std(long_window, center=True).alias(f"{c}_std_1s")
            for c in cols_to_smooth
        ] + [
            pl.col("at__body_center__body_center__dist").rolling_min(long_window, center=True).alias("dist_min_1s"),
            pl.col("at__body_center__body_center__dist").rolling_max(long_window, center=True).alias("dist_max_1s"),
        ])

        # ========== 第五阶段：Lag/Lead特征 ==========
        lag_lead_exprs = []
        for lag in [5, 10, 15, 30]:
            lag_lead_exprs.extend([
                pl.col("at__body_center__body_center__dist").shift(lag).alias(f"dist_lag_{lag}"),
                pl.col("at__body_center__body_center__dist").shift(-lag).alias(f"dist_lead_{lag}"),
                pl.col("approach_speed_500ms").shift(lag).alias(f"approach_lag_{lag}"),
                pl.col("approach_speed_500ms").shift(-lag).alias(f"approach_lead_{lag}"),
                pl.col("agent__body_center__speed_500ms").shift(lag).alias(f"agent_speed_lag_{lag}"),
                pl.col("agent__body_center__speed_500ms").shift(-lag).alias(f"agent_speed_lead_{lag}"),
            ])

        features = features.with_columns(lag_lead_exprs)

        # ========== 第六阶段：趋势特征 ==========
        features = features.with_columns(
            (pl.col("dist_lag_30") - pl.col("at__body_center__body_center__dist")).alias("dist_change_1s"),
            (pl.col("approach_speed_500ms_avg_1s") < -0.5).cast(pl.Float32).alias("approaching_trend"),
        )

        result_element = result_element.join(
            features,
            on=["video_frame", "agent_mouse_id", "target_mouse_id"],
            how="left",
        )
        result.append(result_element)

    return pl.concat(result, how="vertical")

In [ ]:
def preprocess_tracking(df: pl.DataFrame, metadata: dict) -> pl.DataFrame:
    """
    针对宽表格式 (agent_x_nose, target_x_ear_left...) 进行鲁棒清洗
    """
    fps = metadata["frames_per_second"]
    pix_per_cm = metadata["pix_per_cm_approx"]

    # --- 参数设置 ---
    # 1. 形态学阈值: 任何部位距离 Body Center 超过 15cm 视为异常 (老鼠没那么大)
    MAX_DIST_FROM_CENTER_CM = 15.0
    max_dist_px = MAX_DIST_FROM_CENTER_CM * pix_per_cm

    # 2. 速度阈值: 单帧移动超过 100cm/s (瞬移) 视为异常
    MAX_SPEED_CM_S = 100.0
    max_step_px = (MAX_SPEED_CM_S * pix_per_cm) / fps

    # 识别所有坐标列
    x_cols = [c for c in df.columns if "_x_" in c]
    y_cols = [c for c in df.columns if "_y_" in c]

    # 暂存处理表达式
    df_cleaned = df.clone()

    # ===========================
    # Stage 1: 基于身体中心的形态学过滤 (最稳健)
    # ===========================
    # 这种方法比 IQR 更适合 Pose Estimation，因为 IQR 会受到老鼠在场地位置的影响
    prefixes = set([c.split("_x_")[0] for c in x_cols]) # {'agent'} or {'agent', 'target'}

    for prefix in prefixes:
        center_x_col = f"{prefix}_x_body_center"
        center_y_col = f"{prefix}_y_body_center"

        # 如果连身体中心都没有，无法进行此步检查
        if center_x_col not in df.columns:
            continue

        curr_x_cols = [c for c in x_cols if c.startswith(prefix) and "body_center" not in c]

        for xc in curr_x_cols:
            yc = xc.replace("_x_", "_y_")

            # 计算该部位到身体中心的距离
            dist_to_center = (
                (pl.col(xc) - pl.col(center_x_col)).pow(2) +
                (pl.col(yc) - pl.col(center_y_col)).pow(2)
            ).sqrt()

            # 超过阈值置为 None
            df_cleaned = df_cleaned.with_columns([
                pl.when(dist_to_center > max_dist_px).then(None).otherwise(pl.col(xc)).alias(xc),
                pl.when(dist_to_center > max_dist_px).then(None).otherwise(pl.col(yc)).alias(yc)
            ])

    # ===========================
    # Stage 2: 基于速度的瞬移过滤
    # ===========================
    # 对 Body Center 和其他关键点做速度检查
    # 注意：这里我们对所有点都做检查，防止插值前的极端跳变
    for xc in x_cols:
        yc = xc.replace("_x_", "_y_")

        # 计算当前帧与上一帧的距离 (欧氏距离)
        # diff() 计算的是 (t) - (t-1)
        step_dist = ((pl.col(xc).diff()).pow(2) + (pl.col(yc).diff()).pow(2)).sqrt()

        # 如果单步跳变太大，认为当前帧是异常值
        # 注意：这里简单处理，将当前帧置空。更复杂的可以用 shift(-1) 比较
        is_jump = step_dist > max_step_px

        df_cleaned = df_cleaned.with_columns([
            pl.when(is_jump).then(None).otherwise(pl.col(xc)).alias(xc),
            pl.when(is_jump).then(None).otherwise(pl.col(yc)).alias(yc)
        ])

    # ===========================
    # Stage 3: 插值与平滑 (修复 Null)
    # ===========================
    # 1. 线性插值填补刚才产生的 None 以及原始的 NaN
    df_cleaned = df_cleaned.with_columns([
        pl.col(c).interpolate().fill_null(strategy="mean") for c in x_cols + y_cols
    ])

    # 2. Savitzky-Golay 平滑的替代品：Rolling Mean
    # 用于消除微小的抖动噪声
    df_cleaned = df_cleaned.with_columns([
        pl.col(c).rolling_mean(window_size=7, center=True).fill_null(strategy="mean")
        for c in x_cols + y_cols
    ])

    return df_cleaned

In [ ]:
def robustify(submission: pl.DataFrame, dataset: pl.DataFrame, train_test: str = "train"):
    traintest_directory = INPUT_DIR / f"{train_test}_tracking"

    old_submission = submission.clone()
    submission = submission.filter(pl.col("start_frame") < pl.col("stop_frame"))
    if len(submission) != len(old_submission):
        print("ERROR: Dropped frames with start >= stop")

    old_submission = submission.clone()
    group_list = []
    for _, group in submission.group_by("video_id", "agent_id", "target_id"):
        group = group.sort("start_frame")
        mask = np.ones(len(group), dtype=bool)
        last_stop_frame = 0
        for i, row in enumerate(group.rows(named=True)):
            if row["start_frame"] < last_stop_frame:
                mask[i] = False
            else:
                last_stop_frame = row["stop_frame"]
        group_list.append(group.filter(pl.Series("mask", mask)))

    submission = pl.concat(group_list)

    if len(submission) != len(old_submission):
        print("ERROR: Dropped duplicate frames")

    s_list = []
    for row in dataset.rows(named=True):
        lab_id = row["lab_id"]
        video_id = row["video_id"]
        if row["behaviors_labeled"] is None:
            continue

        if video_id in submission.get_column("video_id").to_list():
            continue

        if isinstance(row["behaviors_labeled"], str):
            continue

        print(f"Video {video_id} has no predictions.")

        path = traintest_directory / f"/{lab_id}/{video_id}.parquet"
        vid = pd.read_parquet(path)

        vid_behaviors = json.loads(row["behaviors_labeled"])
        vid_behaviors = sorted(list({b.replace("'", "") for b in vid_behaviors}))
        vid_behaviors = [b.split(",") for b in vid_behaviors]
        vid_behaviors = pd.DataFrame(vid_behaviors, columns=["agent", "target", "action"])

        start_frame = vid.video_frame.min()
        stop_frame = vid.video_frame.max() + 1

        for (agent, target), actions in vid_behaviors.groupby(["agent", "target"]):
            batch_length = int(np.ceil((stop_frame - start_frame) / len(actions)))
            for i, action_row in enumerate(actions.itertuples(index=False)):
                batch_start = start_frame + i * batch_length
                batch_stop = min(batch_start + batch_length, stop_frame)
                s_list.append((video_id, agent, target, action_row["action"], batch_start, batch_stop))

    if len(s_list) > 0:
        submission = pd.concat(
            [
                submission,
                pd.DataFrame(s_list, columns=["video_id", "agent_id", "target_id", "action", "start_frame", "stop_frame"]),
            ]
        )
        print("ERROR: Filled empty videos")

    return submission

In [ ]:
(WORKING_DIR / "self_features").mkdir(exist_ok=True, parents=True)
(WORKING_DIR / "pair_features").mkdir(exist_ok=True, parents=True)

rows = test_dataframe.rows(named=True)

for row in tqdm(rows, total=len(rows)):
    lab_id = row["lab_id"]
    video_id = row["video_id"]

    tracking_path = TEST_TRACKING_DIR / f"{lab_id}/{video_id}.parquet"
    tracking = pl.read_parquet(tracking_path)

    self_features = make_self_features(metadata=row, tracking=tracking)
    pair_features = make_pair_features(metadata=row, tracking=tracking)

    self_features.write_parquet(WORKING_DIR / "self_features" / f"{video_id}.parquet")
    pair_features.write_parquet(WORKING_DIR / "pair_features" / f"{video_id}.parquet")

    del self_features, pair_features
    gc.collect()

In [ ]:
# ==========================================
# 1. Define the Lookup Dictionary
# ==========================================
MIN_DURATION_LOOKUP = {
    'CRIM13_attack': 5,
    'CRIM13_sniff': 0,
    'CRIM13_disengage': 10,
    'CRIM13_approach': 5,
    'CRIM13_rear': 6,
    'CRIM13_selfgroom': 5,
    'CRIM13_mount': 5,
    'DeliriousFly_sniff': 6,
    'DeliriousFly_dominance': 7,
    'DeliriousFly_attack': 11,
    'UppityFerret_huddle': 5,
    'UppityFerret_reciprocalsniff': 2,
    'UppityFerret_sniffgenital': 2,
    'TranquilPanther_sniff': 2,
    'TranquilPanther_sniffgenital': 2,
    'TranquilPanther_rear': 8,
    'TranquilPanther_mount': 9,
    'TranquilPanther_intromit': 18,
    'TranquilPanther_selfgroom': 8,
    'LyricalHare_attack': 10,
    'LyricalHare_escape': 6,
    'LyricalHare_defend': 10,
    'LyricalHare_approach': 38,
    'LyricalHare_freeze': 10,
    'LyricalHare_sniff': 10,
    'LyricalHare_rear': 12,
    'ElegantMink_sniff': 5,
    'ElegantMink_mount': 18,
    'ElegantMink_intromit': 49,
    'ElegantMink_allogroom': 38,
    'ElegantMink_ejaculate': 151,
    'ElegantMink_attemptmount': 9,
    'ElegantMink_attack': 8,
    'CalMS21_supplemental_sniff': 2,
    'CalMS21_supplemental_mount': 4,
    'CalMS21_supplemental_intromit': 18,
    'CalMS21_supplemental_sniffgenital': 2,
    'CalMS21_supplemental_attack': 2,
    'CalMS21_supplemental_dominancemount': 8,
    'CalMS21_supplemental_sniffface': 2,
    'CalMS21_supplemental_sniffbody': 2,
    'CalMS21_supplemental_approach': 2,
    'CalMS21_supplemental_attemptmount': 9,
    'SparklingTapir_attack': 4,
    'SparklingTapir_defend': 2,
    'SparklingTapir_escape': 2,
    'SparklingTapir_mount': 10,
    'NiftyGoldfinch_approach': 4,
    'NiftyGoldfinch_sniffface': 2,
    'NiftyGoldfinch_sniff': 2,
    'NiftyGoldfinch_defend': 2,
    'NiftyGoldfinch_rear': 4,
    'NiftyGoldfinch_climb': 6,
    'NiftyGoldfinch_flinch': 3,
    'NiftyGoldfinch_escape': 3,
    'NiftyGoldfinch_selfgroom': 6,
    'NiftyGoldfinch_follow': 8,
    'NiftyGoldfinch_exploreobject': 5,
    'NiftyGoldfinch_chase': 5,
    'NiftyGoldfinch_attack': 4,
    'NiftyGoldfinch_sniffgenital': 3,
    'NiftyGoldfinch_tussle': 11,
    'NiftyGoldfinch_biteobject': 21,
    'NiftyGoldfinch_dig': 6,
    'GroovyShrew_sniff': 2,
    'GroovyShrew_approach': 3,
    'GroovyShrew_rear': 5,
    'GroovyShrew_sniffgenital': 4,
    'GroovyShrew_climb': 10,
    'GroovyShrew_run': 10,
    'GroovyShrew_selfgroom': 8,
    'GroovyShrew_escape': 6,
    'GroovyShrew_dig': 8,
    'GroovyShrew_rest': 25,
    'GroovyShrew_defend': 5,
    'GroovyShrew_attemptmount': 13,
    'BoisterousParrot_shepherd': 15,
    'AdaptableSnail_avoid': 9,
    'AdaptableSnail_chase': 2,
    'AdaptableSnail_attack': 2,
    'AdaptableSnail_chaseattack': 4,
    'AdaptableSnail_rear': 7,
    'AdaptableSnail_approach': 7,
    'AdaptableSnail_submit': 5,
    'CalMS21_task1_sniffbody': 2,
    'CalMS21_task1_approach': 4,
    'CalMS21_task1_sniffface': 3,
    'CalMS21_task1_sniffgenital': 2,
    'CalMS21_task1_mount': 4,
    'CalMS21_task1_genitalgroom': 8,
    'CalMS21_task1_sniff': 2,
    'CalMS21_task1_attack': 4,
    'CalMS21_task1_intromit': 56,
    'PleasantMeerkat_escape': 9,
    'PleasantMeerkat_attack': 9,
    'PleasantMeerkat_chase': 9,
    'PleasantMeerkat_follow': 45,
    'JovialSwallow_sniff': 6,
    'JovialSwallow_attack': 9,
    'JovialSwallow_chase': 9,
    'InvincibleJellyfish_sniffgenital': 2,
    'InvincibleJellyfish_sniff': 2,
    'InvincibleJellyfish_escape': 11,
    'InvincibleJellyfish_dig': 20,
    'InvincibleJellyfish_dominancegroom': 14,
    'InvincibleJellyfish_selfgroom': 10,
    'InvincibleJellyfish_attack': 3,
    'InvincibleJellyfish_allogroom': 13,
    'CalMS21_task2_attack': 2,
    'CalMS21_task2_sniff': 2,
    'CalMS21_task2_mount': 3,
    'ReflectiveManatee_sniff': 4,
    'ReflectiveManatee_attack': 4,
    'CautiousGiraffe_sniff': 4,
    'CautiousGiraffe_sniffgenital': 5,
    'CautiousGiraffe_reciprocalsniff': 4,
    'CautiousGiraffe_sniffbody': 6,
    'CautiousGiraffe_chase': 19,
    'CautiousGiraffe_escape': 9
}

DEFAULT_MIN_LEN = 5  # Default value if lab+behavior is not in dictionary
GAP_FILL_FRAMES = 10 

In [ ]:
# ... (前面的特征生成代码保持不变) ...

# ==========================================
# 1. 修复后的推理主循环
# ==========================================

from scipy.ndimage import gaussian_filter1d

# 确保 MIN_DURATION_LOOKUP 完整 (这里只列出部分，请确保你的代码里是完整的)
# MIN_DURATION_LOOKUP = { ... } 
# DEFAULT_MIN_LEN = 5  
# GAP_FILL_FRAMES = 10 

group_submissions = []
# 按 (实验室, 视频, 施动者, 目标) 分组处理
groups = list(test_behavior_dataframe.group_by("lab_id", "video_id", "agent", "target", maintain_order=True))

for (lab_id, video_id, agent, target), group in tqdm(groups, total=len(list(groups))):
    # 解析 agent 和 target 的 ID
    agent_mouse_id = int(re.search(r"mouse(\d+)", agent).group(1))
    target_mouse_id = -1 if target == "self" else int(re.search(r"mouse(\d+)", target).group(1))

    # 确定特征文件路径 (self 或 pair)
    feature_dir = WORKING_DIR / ("self_features" if target == "self" else "pair_features")
    feature_path = feature_dir / f"{video_id}.parquet"
    
    # 懒加载并过滤出当前的一对老鼠的数据
    data_scan = pl.scan_parquet(feature_path).filter(
        (pl.col("agent_mouse_id") == agent_mouse_id) & 
        (pl.col("target_mouse_id") == target_mouse_id)
    )
    
    # 将数据加载到内存
    try:
        index = data_scan.select(INDEX_COLS).collect()
        feature = data_scan.select(pl.exclude(INDEX_COLS)).collect()
    except Exception as e:
        print(f"Skipping {video_id} due to load error: {e}")
        continue

    prediction_dataframe = index.clone()
    has_valid_prediction = False
    
    # 用于存储概率列名和对应的平均阈值
    prob_cols = []
    behavior_thresholds = {}
    
    # 【修复1】: 使用集合防止重复处理同一行为
    processed_behaviors = set()
    # 【修复2】: 收集所有预测Series，稍后一次性加入DF，提高性能并防止中途修改出错
    pred_series_list = []

    # --- 1. 收集预测结果 & 计算平均阈值 ---
    for row in group.rows(named=True):
        behavior = row["behavior"]
        
        # 如果该行为已经处理过，直接跳过 (解决 duplicate column 错误的核心)
        if behavior in processed_behaviors:
            continue
        
        # 查找该行为的所有 Fold 模型
        fold_dirs = list((WORKING_DIR / "results" / lab_id / behavior).glob("fold_*"))
        if not fold_dirs:
            continue
            
        # 标记为已处理
        processed_behaviors.add(behavior)

        fold_preds_list = []
        fold_thresholds_list = [] 
        
        for fold_dir in fold_dirs:
            # 1. 读取【当前 Fold】的阈值
            current_fold_thresh = 0.5 
            if (fold_dir / "threshold.txt").exists():
                with open(fold_dir / "threshold.txt", "r") as f:
                    current_fold_thresh = float(f.read().strip())
            fold_thresholds_list.append(current_fold_thresh)

            # 2. 加载模型并预测
            model = xgb.Booster(model_file=fold_dir / "model.json")
            try:
                model.set_param({"device": "cuda"}) 
            except xgb.core.XGBoostError:
                model.set_param({"device": "cpu"})
            
            # 使用 DMatrix 进行预测 
            dtest = xgb.DMatrix(feature, feature_names=feature.columns)
            fold_predictions = model.predict(dtest)
            
            fold_preds_list.append(fold_predictions)
            
            # 手动清理内存
            del model, dtest, fold_predictions

        if fold_preds_list:
            has_valid_prediction = True
            
            # A. 软投票 (Soft Voting)
            avg_pred = np.mean(fold_preds_list, axis=0)
            
            # B. 阈值平均
            avg_threshold = np.mean(fold_thresholds_list)
            
            # 定义概率列的名称
            behavior_key = f"{behavior}_prob"
            behavior_thresholds[behavior_key] = avg_threshold
            
            # 【修复2】: 添加到列表，而不是直接修改 DF
            pred_series_list.append(pl.Series(name=behavior_key, values=avg_pred, dtype=pl.Float32))
            prob_cols.append(behavior_key)

    if not prob_cols or not has_valid_prediction:
        continue

    # 【修复2】: 一次性将所有原始概率列加入 DataFrame
    prediction_dataframe = prediction_dataframe.with_columns(pred_series_list)

    # --- 2. 概率平滑 (高斯滤波) ---
    prediction_dataframe = prediction_dataframe.sort("video_frame")
    smoothed_cols = []
    
    for c in prob_cols:
        arr = prediction_dataframe.get_column(c).to_numpy()
        # 应用高斯平滑 (sigma=2.0)
        smoothed = gaussian_filter1d(arr, sigma=2.0) 
        # 注意：这里 Series 名字和原列名一样，with_columns 会自动替换旧列
        smoothed_cols.append(pl.Series(c, smoothed))
        
    prediction_dataframe = prediction_dataframe.with_columns(smoothed_cols)

    # --- 3. 归一化打分 & 竞争机制 (强制单标签) ---
    score_exprs = []
    behavior_names = []
    
    for col_name in prob_cols:
        thresh = behavior_thresholds[col_name]
        behavior_name = col_name.replace("_prob", "")
        behavior_names.append(behavior_name)
        
        # Score = Prob / Threshold
        score_col = (pl.col(col_name) / thresh)
        
        # 没过阈值得分为0
        score_exprs.append(
            pl.when(pl.col(col_name) < thresh)
            .then(pl.lit(0.0))
            .otherwise(score_col)
            .alias(f"score_{behavior_name}")
        )
        
    # 计算得分
    scores_df = prediction_dataframe.with_columns(score_exprs)
    score_col_names = [f"score_{b}" for b in behavior_names]
    
    # 找出赢家
    scores_df = scores_df.with_columns(
        pl.concat_list(score_col_names).alias("all_scores")
    ).with_columns(
        pl.col("all_scores").list.arg_max().alias("best_idx"),
        pl.col("all_scores").list.max().alias("best_score")
    )
    
    # 映射回名称
    label_expr = pl.lit("none")
    for idx, name in enumerate(behavior_names):
        label_expr = pl.when(pl.col("best_idx") == idx).then(pl.lit(name)).otherwise(label_expr)
    
    # 如果所有得分都为0，则是none
    label_expr = pl.when(pl.col("best_score") == 0.0).then(pl.lit("none")).otherwise(label_expr)
    
    prediction_labels_dataframe = scores_df.select(
        INDEX_COLS + [label_expr.alias("prediction")]
    )

    # --- 4. 转换为 RLE ---
    group_submission = (
        prediction_labels_dataframe
        .filter((pl.col("prediction") != pl.col("prediction").shift(1))) 
        .with_columns(pl.col("video_frame").shift(-1).alias("stop_frame")) 
        .filter(pl.col("prediction") != "none") 
        .select(
            pl.col("video_id"),
            ("mouse" + pl.col("agent_mouse_id").cast(str)).alias("agent_id"),
            pl.when(pl.col("target_mouse_id") == -1)
            .then(pl.lit("self"))
            .otherwise("mouse" + pl.col("target_mouse_id").cast(str))
            .alias("target_id"),
            pl.col("prediction").alias("action"),
            pl.col("video_frame").alias("start_frame"),
            pl.col("stop_frame"),
        )
    )


    # --- 5. 基于最小时长的后处理过滤 ---
    valid_rows = []
    if len(group_submission) > 0:
        # 获取视频最大帧数以处理最后一行的 None
        end_frame = prediction_labels_dataframe.select(pl.col("video_frame").max()).item()
        
        for row in group_submission.rows(named=True):
            behavior = row["action"]
            stop = row["stop_frame"] if row["stop_frame"] is not None else (end_frame + 1)
            duration = stop - row["start_frame"]
            
            key = f"{lab_id}_{behavior}"
            min_len = MIN_DURATION_LOOKUP.get(key, DEFAULT_MIN_LEN)
            
            if duration >= min_len:
                valid_rows.append(row)
        
        if valid_rows:
            group_submission = pl.DataFrame(valid_rows, schema=group_submission.schema)
        else:
            group_submission = group_submission.clear()

    group_submissions.append(group_submission)
    
    del feature, index, prediction_dataframe, scores_df, prediction_labels_dataframe
    gc.collect()

In [ ]:
# ... [Saving submission code remains unchanged] ...
submission = pl.concat(group_submissions, how="vertical").sort(
    "video_id",
    "agent_id",
    "target_id",
    "action",
    "start_frame",
    "stop_frame",
)
submission = robustify(submission, test_dataframe, train_test="test")
submission.with_row_index("row_id").write_csv("/kaggle/working/submission.csv")